# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to use the `mlcroissant` library to load, explore, and process the [FAIR^2](https://doi.org/10.71728/senscience.vq4a-28xa) mass spectrometry dataset. All dataset entities (record sets, fields, columns, etc.) are referenced by their `@id` for clarity and reproducibility.

### Dataset Source
The dataset Croissant schema is publicly available at:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load and inspect dataset metadata using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

# Print key metadata attributes
print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Version: {getattr(metadata, 'version', 'N/A')}")
print(f"Published: {getattr(metadata, 'datePublished', 'N/A')}")
print(f"License: {getattr(metadata, 'license', 'N/A')}")
print(f"Keywords: {getattr(metadata, 'keywords', [])}")

## 2. Data Overview
List available record sets, their fields and columns, referencing all entities by their `@id`.

> **Note:** The `mlcroissant` interface exposes `dataset.record_sets` and each `record_set` has an `@id`, a name, and a set of fields. Each field has an `@id`, a name, data type, and may be linked to specific file columns (also referenced by `@id`).

In [ ]:
# List record sets and their fields
print("Available record sets:")
record_sets = dataset.record_sets
record_set_ids = []
for rs in record_sets:
    print(f"  - RecordSet @id: {rs.id}")
    print(f"    Name: {rs.name}")
    if hasattr(rs, 'description') and rs.description:
        print(f"    Description: {rs.description}")
    print(f"    Fields:")
    for field in rs.fields:
        print(f"      • Field @id: {field.id}")
        print(f"        Name: {field.name}")
        print(f"        Data type: {getattr(field, 'dataType', None)}")
        if hasattr(field, 'columns'):
            for column in field.columns:
                print(f"        Column @id: {column.id}")
        print()
    record_set_ids.append(rs.id)
print("\nRecord sets found:", record_set_ids)

## 3. Data Extraction
Load data from each record set into a Pandas DataFrame for analysis.

- Use the `recordSet` and `field`/`column` `@id` values from the previous overview.
- Each record set is referenced for extraction using its `@id`.

In [ ]:
# Extract each record set to a pandas DataFrame
dataframes = {}
for record_set_id in record_set_ids:
    print(f"Extracting records from record set @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        print(f"  Columns: {list(df.columns)}")
        print(f"  {len(df)} records loaded.")
        dataframes[record_set_id] = df
    else:
        print("  No records found.")
# For demonstration, pick the first record set for further analysis
if record_set_ids:
    main_record_set_id = record_set_ids[0]
    main_df = dataframes[main_record_set_id]
    print("\nSample of extracted data:")
    display(main_df.head())

## 4. Exploratory Data Analysis (EDA)
Apply EDA: filter numeric columns, normalize a numeric field, and group by a categorical field. All fields referenced using their `@id` (or column name as provided in data extraction).

> Adjust the `numeric_field_id` and `group_field_id` variables below based on the available fields.

In [ ]:
# -- Set analysis parameters (Replace below with appropriate @id/column for your case) -- #
# Example: If the column for "molecular weight" is 'MW' or '@id: cr:MW', set numeric_field_id = 'MW'
#           If there is a sample group column such as 'Sample' use its name or @id

main_df_columns = list(main_df.columns)
print("Columns in main record set:")
print(main_df_columns)

# Choose a numeric field for filtering/normalization
# For demonstration, let's use the first numeric-looking field found.
numeric_field_id = None
for col in main_df_columns:
    # Heuristic: look for typical quantitative field names
    if any(key in col.lower() for key in ['mw', 'weight', 'count', 'abundance', 'value', 'score', 'area']):
        numeric_field_id = col
        break
if numeric_field_id is None:
    numeric_field_id = main_df_columns[0]  # fallback
print(f"Using numeric field: {numeric_field_id}")

# Try to use a group/categorical field (e.g. 'SampleID', 'Sample', 'Classification')
group_field_id = None
for col in main_df_columns:
    if any(key in col.lower() for key in ['sample', 'group', 'type', 'class', 'category', 'species']):
        group_field_id = col
        break
print(f"Using group field: {group_field_id}")

# Filter rows with numeric_field > threshold (use a reasonable threshold based on sample values)
threshold = main_df[numeric_field_id].dropna().astype(float).mean() if not main_df[numeric_field_id].dropna().empty else 10
filtered_df = main_df[pd.to_numeric(main_df[numeric_field_id], errors='coerce') > threshold]
print(f"\nFiltered records where {numeric_field_id} > {threshold:.2f} (by column @id/name): {len(filtered_df)} rows")
display(filtered_df.head())

# Normalize the selected numeric field
filtered_df.loc[:, f"{numeric_field_id}_normalized"] = (
    pd.to_numeric(filtered_df[numeric_field_id], errors='coerce') - pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').mean()
) / pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Optionally, group by group_field (if present)
if group_field_id and group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id).agg({numeric_field_id: ['mean', 'count']})
    print(f"\nGrouped data by {group_field_id}:")
    display(grouped_df.head())

## 5. Visualization
Plot data distributions and relationships. Visualizations use field `@id` (or column name) directly.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the selected numeric field
plt.figure(figsize=(8,4))
sns.histplot(pd.to_numeric(main_df[numeric_field_id], errors='coerce').dropna(), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel('Frequency')
plt.show()

# If available and sufficient, plot boxplot grouped by group_field
if group_field_id and group_field_id in main_df.columns:
    plt.figure(figsize=(10,5))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=main_df)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
- Successfully loaded dataset metadata and data using the Croissant schema and the `mlcroissant` library.
- Explored record set and field `@id` structure for reproducible reference.
- Performed basic EDA: filtered by a numeric field, performed normalization, grouped by a categorical field, and visualized distributions.

**Next steps:**
- Deepen analysis based on project-specific questions (e.g. protein biomarker selection).
- Join with other Croissant-compliant datasets as needed.

*All processing steps referenced dataset entities via their `@id` for full FAIRness.*